# A periodogram from scratch

A PROTOTYPE of the compute gate (Prompt 268, COMPUTE_GATE_II.md): it is not in the catalogue and not shipped. It reads its input as the text `gravitas_input` and its settings as the JSON text `gravitas_params`, and returns `gravitas_output`, a dict of numbers. Python only, numpy and scipy only.


In [ ]:
import json
import numpy as np
doc = json.loads(gravitas_input)
assert doc.get("format") == "gravitas.observation" and doc.get("kind") == "time-series", "this notebook reads a gravitas.observation/1 time series"
params = json.loads(gravitas_params) if gravitas_params else {}
col = {c["id"]: np.array(c["values"], float) for c in doc["columns"]}
t, y = col["time"], col["flux"]
assert len(t) >= 10, "refused: fewer than 10 points"
assert bool(np.all(np.diff(t) > 0)), "refused: the times are not strictly increasing"
yy = float(((y - y.mean()) ** 2).sum())
assert yy > 0, "refused: the flux is constant, there is no variance to explain"
P = np.array(params["periods"], float) if params.get("periods") else 0.5 * 20.0 ** (np.arange(400) / 399)
assert bool(np.all(P > 0)), "refused: a trial period is not positive"

In [ ]:
tm = t.mean()
power = np.zeros(len(P))
for k, period in enumerate(P):
    wk = 2 * np.pi / period
    A = np.column_stack([np.cos(wk * (t - tm)), np.sin(wk * (t - tm)), np.ones(len(t))])
    beta = np.linalg.solve(A.T @ A, A.T @ y)
    r = y - A @ beta
    power[k] = 1.0 - float(r @ r) / yy
peak = int(np.argmax(power))
gravitas_output = {"answer": {"periods": P.tolist(), "power": power.tolist(), "peakIndex": peak, "peakPeriod": float(P[peak]), "peakPower": float(power[peak])}}